# Reading from a CSV

Reference: [Original source](https://tutswiki.com/pandas-cookbook/chapter1/)

**Learning outcomes**

- Load a CSV file into a pandas `DataFrame` with `read_csv`, and explain what each option you pass does.
- Describe the difference between a `DataFrame`, a `Series` and an index.
- Inspect a new dataset with `shape`, `dtypes`, `info()`, `describe()` and `head()`.
- Select one or several columns and plot them with readable titles and labels.

**Before you start:** recall from Module 2 that a *dictionary* maps keys to values and a *list* holds values in order. A `DataFrame` is very much like a dictionary of columns, where every column is a list-like sequence of the same length.

## Setup

These notebooks are tested with **pandas 3.x**. Older versions (1.x, 2.x) behave slightly differently in places, so if your output looks different from the `You should see` notes, check your version first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from course_checks import check, check_equal

print(pd.__version__)

**You should see:** a version starting with `3.`, for example `3.1.1`.

## Reading the file

You can read data from a CSV file using the `read_csv` function.

We're going to be looking at some [cyclist data from Montréal](https://donnees.ville.montreal.qc.ca/dataset/velos-comptage). We're using the data from 2012: one row per day, one column per bike counter, and the value is the number of cyclists counted.

In [ ]:
sample_df = pd.read_csv('bikes.csv', sep=';', encoding='latin1', parse_dates=['Date'], dayfirst=True, index_col='Date')

# Look at the first 3 rows
sample_df[:3]

**You should see:** a table of 3 rows. The first column, `Date`, is printed in bold on the left: that is the *index*. The dates are `2012-01-01`, `2012-01-02`, `2012-01-03`. The `Berri 1` column holds `35`, `83`, `135`. Some columns contain `NaN`, which means *Not a Number*: pandas' marker for a missing value.

## Why do we need all those options?

That one line passed five arguments to `read_csv`. Let's remove them one at a time and see what goes wrong. You will understand each option much better after you have seen the problem it solves.

### `sep`: what separates the columns?

A CSV is a *comma*-separated file by default, but this one uses semicolons (`;`), as is common in French-speaking regions where the comma is the decimal mark. If you do not say so, pandas does not find any column boundary.

In [ ]:
no_sep = pd.read_csv('bikes.csv')
print(no_sep.shape)
no_sep.head(2)

**You should see:** `(310, 1)`: 310 rows but only **one** column, whose name is the whole header line glued together with semicolons. With `sep=';'` we get 10 columns (9 counters plus `Date`), because pandas now knows where to cut.

### `encoding`: how are characters stored as bytes?

A file is just bytes. An *encoding* is the table that says which byte means which character. Today most files use UTF-8, but older files from Windows or French systems often use `latin1` (also called ISO-8859-1), where `é` is a single byte. If you decode with the wrong table, accented letters turn into garbage, or pandas refuses to read the file.

The original Montréal file is `latin1`, and its columns are called `Berri 1`, `Brébeuf`, `Côte-Sainte-Catherine`... **Honest note:** the copy of `bikes.csv` shipped with this course has already lost its accents (they are stored as a literal `?`, and the file is plain ASCII). So here `utf-8` and `latin1` give the *same* result. To see the real problem, we build a tiny latin1 file in memory:

In [ ]:
import io

raw_bytes = 'Brébeuf;Côte-Sainte-Catherine\n1;2\n'.encode('latin1')
print(raw_bytes)

# 1. Wrong table: read the latin1 bytes as UTF-8 (pandas' default)
try:
    pd.read_csv(io.BytesIO(raw_bytes), sep=';')
except UnicodeDecodeError as err:
    print('utf-8 failed:', err.reason)

# 2. Right table
print(pd.read_csv(io.BytesIO(raw_bytes), sep=';', encoding='latin1').columns.tolist())

# 3. The opposite mistake: UTF-8 bytes read as latin1 -> "mojibake"
utf8_bytes = 'Brébeuf;Côte-Sainte-Catherine\n1;2\n'.encode('utf-8')
print(pd.read_csv(io.BytesIO(utf8_bytes), sep=';', encoding='latin1').columns.tolist())

**You should see:**

```text
b'Br\xe9beuf;C\xf4te-Sainte-Catherine\n1;2\n'
utf-8 failed: invalid continuation byte
['Brébeuf', 'Côte-Sainte-Catherine']
['BrÃ©beuf', 'CÃ´te-Sainte-Catherine']
```

(The exact wording of the error reason can differ slightly.) So with a wrong encoding you get either an error or garbled accents like `Ã©`. When you meet that, try `encoding='latin1'` or ask who produced the file.

### `parse_dates`: turn text into real dates

Without it, the `Date` column stays plain text. You cannot sort it chronologically, subtract two dates or ask for the weekday.

In [ ]:
no_dates = pd.read_csv('bikes.csv', sep=';', encoding='latin1')
print(no_dates['Date'].dtype)
no_dates['Date'].head(3)

**You should see:** a dtype of `str` (pandas 3 calls its text type `str`; older versions say `object`) and the values `01/01/2012`, `02/01/2012`, `03/01/2012`.

### `dayfirst`: is `02/01/2012` the 2nd of January or the 1st of February?

Dates like `02/01/2012` are ambiguous. In Canada they are written day/month/year, but pandas assumes month first unless told otherwise. Let's see what happens to `parse_dates` with and without `dayfirst`.

In [ ]:
month_first = pd.read_csv('bikes.csv', sep=';', encoding='latin1', parse_dates=['Date'])
day_first = pd.read_csv('bikes.csv', sep=';', encoding='latin1', parse_dates=['Date'], dayfirst=True)

print('without dayfirst:', month_first['Date'].dtype, '->', month_first['Date'][1])
print('with dayfirst:   ', day_first['Date'].dtype, '->', day_first['Date'][1])

**You should see:**

```text
without dayfirst: str -> 02/01/2012
with dayfirst:    datetime64[us] -> 2012-01-02 00:00:00
```

Without `dayfirst`, pandas starts reading the dates month-first, hits a value such as `13/01/2012` (there is no 13th month) and gives up: the column silently stays as *text*, with no error. (In other situations a month-first reading can succeed and quietly turn the 2nd of January into the 1st of February.) Moral: always check the `dtype` after `parse_dates`. The exact dtype unit (`us` or `ns`) depends on the pandas version.

### `index_col`: which column labels the rows?

Without `index_col`, pandas labels rows `0, 1, 2, ...` and `Date` is an ordinary column. With `index_col='Date'`, each row is labelled by its date, so we can write `sample_df.loc['2012-06-15']` instead of searching for it.

In [ ]:
no_index = pd.read_csv('bikes.csv', sep=';', encoding='latin1', parse_dates=['Date'], dayfirst=True)
no_index.head(3)

**You should see:** the same rows as before, but the left-hand labels are `0`, `1`, `2` and `Date` is a normal column next to `Berri 1`.

In the rest of this notebook we continue with `sample_df`, which uses all five options.

## What is a DataFrame?

It is simply the table that you know from an Excel spreadsheet. Compared to a list of lists, each column of a DataFrame explicitly has a single data type. Each row is often regarded as one sample (one observation).

Three words to keep apart:

- **DataFrame**: the whole 2-dimensional table (rows and columns).
- **Series**: *one* column (or one row) of a DataFrame: a 1-dimensional sequence of values *with an index attached*.
- **Index**: the labels of the rows. Here they are dates. Rows are found by their label, and the index is what lets pandas line up data from different sources.

Like a dictionary, `sample_df['Berri 1']` looks a column up by its name. Unlike a plain dictionary, all columns share the same index.

In [ ]:
print(type(sample_df))
print(type(sample_df['Berri 1']))
sample_df.index[:3]

**You should see:** `<class 'pandas.DataFrame'>` (older pandas says `pandas.core.frame.DataFrame`), then `<class 'pandas.Series'>`, then a `DatetimeIndex` starting with `2012-01-01`, `2012-01-02`, `2012-01-03`.

## First look at a new dataset

Whenever you load data, ask these questions before anything else: how big is it, what types are the columns, and is anything missing?

In [ ]:
sample_df.shape

**You should see:** `(310, 9)`: 310 rows (days) and 9 columns (counters). The `Date` index is *not* counted as a column.

In [ ]:
sample_df.dtypes

**You should see:** `int64` for most counters (whole numbers) and `float64` for the two columns that are empty (`Brébeuf...` and `St-Urbain...`): a column containing `NaN` must be `float64`.

In [ ]:
sample_df.info()

**You should see:** a `DatetimeIndex` of 310 entries from `2012-01-01` to `2012-11-05`, and for each column the count of *non-null* values: 310 for most, and `0 non-null` for the two empty columns.

In [ ]:
sample_df.describe()

**You should see:** a table of summary statistics (`count`, `mean`, `std`, `min`, `25%`, `50%`, `75%`, `max`) per column. For `Berri 1` the mean is about 2985 cyclists per day and the maximum is 7077.

## Selecting columns

Let's start by showing the different columns available.

In [ ]:
sample_df.columns

We can select one column using the `[]` operator. The result is a `Series`.

In [ ]:
sample_df["Berri 1"]

## Plotting

A column can be visualized with `.plot()`. A plot without a title or axis labels is hard to read, so we add them. `plt.show()` at the end tells matplotlib to draw the figure.

In [ ]:
sample_df["Berri 1"].plot(title='Daily cyclists at the Berri 1 counter, 2012')
plt.ylabel('Cyclists per day')
plt.show()

**You should see:** one line chart. The x-axis is the date (January to early November), the y-axis is cyclists per day. The line is low in winter (a few dozen to a few hundred), rises in spring, is high and noisy in summer (peaks of about 7000), then drops again in autumn.

Note that what we're seeing is the plot of only one column. What if you want multiple columns? Pass a *list* of column names (so you see two pairs of brackets).

In [ ]:
ax = sample_df[["Berri 1", 'Maisonneuve 1']].plot(title='Two counters compared')
ax.set_ylabel('Cyclists per day')
plt.show()

**You should see:** two lines with a legend naming `Berri 1` and `Maisonneuve 1`, both following the same seasonal shape, with `Berri 1` usually higher.

### A deliberately bad chart

To plot the whole dataframe, we can call `plot()` on it directly. Run it, and then ask yourself: **what is wrong with this chart?**

In [ ]:
sample_df.plot()
plt.show()

**You should see:** a busy chart with 9 entries in the legend. Problems: no title, no axis labels, a long legend that covers the lines, two columns (`Brébeuf...`, `St-Urbain...`) that contain no data at all and therefore draw nothing, and too many overlapping lines to compare anything. The fix is to *choose* what to show: fewer columns, a title, labelled axes.

Sometimes one of the best things you can do is look at a *subset*: for example a single month. Because our index is made of dates, a partial date such as `'2012-06'` selects all of June.

In [ ]:
june = sample_df.loc['2012-06', ['Berri 1', 'Maisonneuve 1', 'du Parc']]
ax = june.plot(title='Cyclists per day in June 2012')
ax.set_ylabel('Cyclists per day')
plt.show()

**You should see:** three lines over 30 days with a readable legend. The lines move together up and down, following a weekly rhythm.

### ✏️ Your turn

Select the two columns `'Berri 1'` and `'du Parc'` from `sample_df` into a new DataFrame called `two_counters`. Remember: several columns means a *list* of names.

In [ ]:
two_counters = ...  # <- replace ... with your code

In [ ]:
check_equal(lambda: two_counters.shape, (310, 2), hint="Use double brackets: sample_df[['Berri 1', 'du Parc']]")

<details>
<summary>Show solution</summary>

```python
two_counters = sample_df[['Berri 1', 'du Parc']]
```

</details>

### ✏️ Your turn

On which date did the `Berri 1` counter record the most cyclists, and on which weekday was that? Store the date in `busiest_day` (hint: a Series has an `idxmax()` method that returns the *index label* of the largest value) and the weekday name in `weekday` (a date has a `.day_name()` method).

In [ ]:
busiest_day = ...  # <- replace ...
weekday = ...      # <- replace ...

In [ ]:
check(lambda: str(busiest_day.date()) == '2012-06-15', hint="sample_df['Berri 1'].idxmax()")
check_equal(lambda: weekday, 'Friday', hint='busiest_day.day_name()')

<details>
<summary>Show solution</summary>

```python
busiest_day = sample_df['Berri 1'].idxmax()
weekday = busiest_day.day_name()
```

</details>

### ✏️ Your turn

How many cyclists did the `Berri 1` counter record in total in 2012 (the data runs until 5 November)? Store the number in `total_berri`.

In [ ]:
total_berri = ...  # <- replace ...

In [ ]:
check_equal(lambda: int(total_berri), 925365, hint="A Series has a .sum() method")

<details>
<summary>Show solution</summary>

```python
total_berri = sample_df['Berri 1'].sum()
```

</details>

### ✏️ Your turn

Fix the bad chart: plot only `'Berri 1'` and `'du Parc'`, give the chart a `title`, and store the axes that `.plot()` returns in `ax`. (Add `plt.ylabel(...)` or `ax.set_ylabel(...)` and `plt.show()` as you like.)

In [ ]:
ax = ...  # <- replace ... with sample_df[[...]].plot(title=...)

In [ ]:
check(lambda: len(ax.lines) == 2 and ax.get_title() != '', hint="sample_df[['Berri 1', 'du Parc']].plot(title='...')")

<details>
<summary>Show solution</summary>

```python
ax = sample_df[['Berri 1', 'du Parc']].plot(title='Berri 1 vs du Parc, 2012')
ax.set_ylabel('Cyclists per day')
plt.show()
```

</details>

## ✅ Check yourself

- I can say what `sep`, `encoding`, `parse_dates`, `dayfirst` and `index_col` do, and what goes wrong without them.
- I can explain the difference between a DataFrame, a Series and an index.
- I can use `shape`, `dtypes`, `info()` and `describe()` to get to know a new dataset.
- I know that a wrong encoding gives an error or garbled accents, and that `NaN` means a missing value.
- I can plot a column with a title and axis labels, and I can tell why `sample_df.plot()` on 9 columns is a poor chart.

## Next

In the next notebook, `09_Select`, we load a much larger dataset (111,000 rows of New York 311 complaints) and learn precisely how to pick rows and columns with `[]`, `.loc` and `.iloc`.